# 03 — Gold Aggregate: Fleet & Route Performance Analytics
Builds six analytical Gold tables from the Silver layer.

| Gold Table | Description |
|---|---|
| gold_fleet_summary | Per-vehicle KPIs: deliveries, on-time rate, avg load |
| gold_delivery_performance | Daily delivery counts, on-time rate, avg delay |
| gold_route_analysis | Per-route utilisation, avg delay, total toll cost |
| gold_weekly_trends | Week-over-week delivery volume and delay trends |
| gold_late_delivery_alerts | Deliveries with delay > 1 h (operational alert feed) |
| gold_depot_scorecards | Per-depot fleet size, fuel cost, avg efficiency |

In [ ]:
LAKEHOUSE_NAME = "transportation_lakehouse"
SILVER = "silver"
GOLD   = "gold"

from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE_NAME}.{GOLD}")

sv = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_vehicles")
sr = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_routes")
sd = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_deliveries")
sf = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_fuel_logs")

In [ ]:
# ── gold_fleet_summary ───────────────────────────────────────────────────────
gold_fleet = (
    sd.join(sv.select("vehicle_id", "vehicle_type", "depot", "capacity_tonnes", "is_active"), on="vehicle_id", how="left")
    .groupBy("vehicle_id", "vehicle_type", "depot", "capacity_tonnes", "is_active")
    .agg(
        F.count("delivery_id").alias("total_deliveries"),
        F.sum("is_late").alias("late_deliveries"),
        F.round(F.avg("delay_hrs"), 2).alias("avg_delay_hrs"),
        F.round(F.avg("load_utilisation_pct"), 1).alias("avg_load_utilisation_pct"),
        F.round(F.sum("distance_km"), 0).alias("total_km_driven"),
    )
    .withColumn("on_time_rate_pct", F.round(
        (1 - F.col("late_deliveries") / F.col("total_deliveries")) * 100, 1))
    .withColumn("_updated_at", F.current_timestamp())
)
gold_fleet.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_fleet_summary")
print(f"gold_fleet_summary: {gold_fleet.count():,}")

In [ ]:
# ── gold_delivery_performance ────────────────────────────────────────────────
gold_dp = (
    sd
    .withColumn("delivery_date", F.to_date("planned_departure"))
    .groupBy("delivery_date")
    .agg(
        F.count("delivery_id").alias("total_deliveries"),
        F.sum("is_late").alias("late_deliveries"),
        F.round(F.avg("delay_hrs"), 2).alias("avg_delay_hrs"),
        F.round(F.avg("load_utilisation_pct"), 1).alias("avg_load_pct"),
    )
    .withColumn("on_time_rate_pct", F.round(
        (1 - F.col("late_deliveries") / F.col("total_deliveries")) * 100, 1))
    .orderBy("delivery_date")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_dp.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_delivery_performance")
print(f"gold_delivery_performance: {gold_dp.count():,}")

In [ ]:
# ── gold_route_analysis ──────────────────────────────────────────────────────
gold_ra = (
    sd
    .join(sr.select("route_id", "origin", "destination", "route_type", "toll_cost_gbp"), on="route_id", how="left")
    .groupBy("route_id", "origin", "destination", "route_type")
    .agg(
        F.count("delivery_id").alias("total_deliveries"),
        F.sum("is_late").alias("late_deliveries"),
        F.round(F.avg("delay_hrs"), 2).alias("avg_delay_hrs"),
        F.round(F.avg("load_utilisation_pct"), 1).alias("avg_load_pct"),
        F.round(F.sum("toll_cost_gbp"), 2).alias("total_toll_cost_gbp"),
    )
    .withColumn("on_time_rate_pct", F.round(
        (1 - F.col("late_deliveries") / F.col("total_deliveries")) * 100, 1))
    .withColumn("_updated_at", F.current_timestamp())
)
gold_ra.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_route_analysis")
print(f"gold_route_analysis: {gold_ra.count():,}")

In [ ]:
# ── gold_weekly_trends ───────────────────────────────────────────────────────
gold_wt = (
    sd
    .withColumn("week_start", F.date_trunc("week", F.to_date("planned_departure")))
    .groupBy("week_start")
    .agg(
        F.count("delivery_id").alias("total_deliveries"),
        F.sum("is_late").alias("late_deliveries"),
        F.round(F.avg("delay_hrs"), 2).alias("avg_delay_hrs"),
        F.round(F.sum("distance_km"), 0).alias("total_distance_km"),
    )
    .withColumn("on_time_rate_pct", F.round(
        (1 - F.col("late_deliveries") / F.col("total_deliveries")) * 100, 1))
    .orderBy("week_start")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_wt.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_weekly_trends")
print(f"gold_weekly_trends: {gold_wt.count():,}")

In [ ]:
# ── gold_late_delivery_alerts ────────────────────────────────────────────────
gold_la = (
    sd.filter(F.col("delay_hrs") > 1)
    .join(sr.select("route_id", "origin", "destination"), on="route_id", how="left")
    .select(
        "delivery_id", "vehicle_id", "route_id", "origin", "destination",
        "planned_departure", "actual_arrival", "delay_hrs", "delay_band", "status"
    )
    .orderBy(F.col("delay_hrs").desc())
    .withColumn("_updated_at", F.current_timestamp())
)
gold_la.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_late_delivery_alerts")
print(f"gold_late_delivery_alerts: {gold_la.count():,}")

In [ ]:
# ── gold_depot_scorecards ────────────────────────────────────────────────────
fleet_by_depot = sv.groupBy("depot").agg(F.count("vehicle_id").alias("fleet_size"),
                                          F.sum("is_active").alias("active_vehicles"))

fuel_by_depot = sf.groupBy("depot").agg(
    F.round(F.sum("total_cost_gbp"), 2).alias("total_fuel_cost_gbp"),
    F.round(F.avg("fuel_efficiency_km_per_litre"), 3).alias("avg_fuel_efficiency"),
)

gold_ds = (
    fleet_by_depot
    .join(fuel_by_depot, on="depot", how="left")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_ds.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_depot_scorecards")
print(f"gold_depot_scorecards: {gold_ds.count():,}")

In [ ]:
print("\n=== Gold Aggregate Complete ===")
gold_tables = [
    "gold_fleet_summary", "gold_delivery_performance", "gold_route_analysis",
    "gold_weekly_trends", "gold_late_delivery_alerts", "gold_depot_scorecards"
]
for tbl in gold_tables:
    cnt = spark.table(f"{LAKEHOUSE_NAME}.{GOLD}.{tbl}").count()
    print(f"  {tbl}: {cnt:,}")